In [ ]:
import os
import shutil
import uuid
import glob
import subprocess
from typing import Dict, List, Tuple, Any

import gradio as gr
from PIL import Image, ImageDraw
import tempfile
import debugpy

import numpy as np

import cv2
import json
import math

# Enviroment Set-up

In [ ]:
using_colab = False

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam2.git'

    !mkdir -p videos
    !wget -P videos https://dl.fbaipublicfiles.com/segment_anything_2/assets/bedroom.zip
    !unzip -d videos videos/bedroom.zip

    !mkdir -p ../checkpoints/
    !wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

## Set-up

In [ ]:
# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

### Loading the SAM 2 video predictor

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

sam2_checkpoint = "../checkpoints/sam2.1_hiera_large.pt"
# sam2_checkpoint = "../checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
# model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

### Utilities

In [ ]:
# -----------------------------
# Utilities
# -----------------------------
def extract_frames(video_path: str, workdir: str) -> List[str]:
    """
    Use ffmpeg to extract frames to workdir/frames/%05d.jpg
    Returns list of frame paths sorted.
    """
    frames_dir = os.path.join(workdir, "frames")
    os.makedirs(frames_dir, exist_ok=True)

    # -vsync 0 keeps all frames; adjust as needed.
    # -qscale:v 2 yields decent JPEG quality; adjust if you want PNG.
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-qscale:v", "2",
        "-vsync", "0",
        os.path.join(frames_dir, "%05d.jpg"),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    frame_paths = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    return frame_paths


def draw_points_on_frame(frame_path: str,
                         obj_points: Dict[str, Dict[str, List[Tuple[int, int, int]]]],
                         ref_points: Dict[str, List[Tuple[int, int, int]]],
                         frame_idx: int) -> Image.Image:
    """
    Overlay points for the given frame_idx.
    - obj_points[object_name]["add"|"remove"] -> list[(x,y,frame_idx)]
    - ref_points["Ref 1"/"Ref 2"] -> list[(x,y,frame_idx)]
    """
    im = Image.open(frame_path).convert("RGB")
    draw = ImageDraw.Draw(im)

    # Objects: add/remove
    for obj_name, buckets in obj_points.items():
        for mode, pts in buckets.items():
            for (x, y, f) in pts:
                if f != frame_idx:
                    continue
                # add 포인트는 원, remove 포인트는 X 마크로 표시
                r = 6
                if mode == "add":
                    draw.ellipse((x - r, y - r, x + r, y + r), outline=(0, 255, 0), width=2)
                else:
                    # draw an X
                    draw.line((x - r, y - r, x + r, y + r), fill=(255, 0, 0), width=2)
                    draw.line((x - r, y + r, x + r, y - r), fill=(255, 0, 0), width=2)

    # Reference points: small squares
    for ref_name, pts in ref_points.items():
        for (x, y, f) in pts:
            if f != frame_idx:
                continue
            r = 5
            draw.rectangle((x - r, y - r, x + r, y + r), outline=(0, 200, 255), width=2)

    return im



# Gradio App Codes

In [ ]:
# -----------------------------
# Gradio App (css styles)
# -----------------------------
CSS = '''
/* Left column: make uploaded video smaller so video -> track button fit in view */
#left_col .gr-video, #left_col video {
  max-height: 180px;
  width: 100%;
  object-fit: contain;
}
#left_col {
  max-height: 90vh;
  overflow: auto;
  padding-right: 8px;
}

/* Right column: use a column flex layout so the progress log stays at the bottom.
   Constrain the frame image so it won't overflow in fullscreen. */
#right_col {
  display: flex;
  flex-direction: column;
  height: 90vh;
  gap: 8px;
}
#right_col .gr-row, #right_col .gr-block {
  flex: 0 0 auto;
}

/* Frame image constraints: limit to 60% of viewport height on fullscreen so tall videos don't break layout */
#frame_img img {
  max-width: 100%;
  max-height: 60vh; /* limit to 60% of viewport height */
  object-fit: contain;
  display: block;
  margin: 0 auto;
}

/* Ensure frame image container itself doesn't push other elements and allows scrolling inside if needed */
#frame_img {
  max-height: 60vh;
  overflow: auto;
}

/* Make textbox (progress log) stick to bottom of right column */
#right_col .gr-textbox {
  margin-top: auto;
}
'''

css_inject = """

<style> /* Target common gradio image/canvas wrappers and the component container by elem_id */ #frame_img, .gr-img, .gr-image, .gradio-image { max-height: 60vh !important; overflow: auto !important; } /* img and canvas inside the frame image component */ #frame_img img, #frame_img canvas, .gr-image img, .gr-image canvas, .gradio-image img, .gradio-image canvas { max-height: 60vh !important; max-width: 100% !important; height: auto !important; object-fit: contain !important; display: block !important; margin: 0 auto !important; } /* Ensure right column keeps layout and progress log stays at bottom */ #right_col { display: flex !important; flex-direction: column !important; height: 90vh !important; } #right_col > .gr-row, #right_col > .gr-block { flex: 0 0 auto !important; } #right_col .gr-textbox { margin-top: auto !important; } </style>
"""

In [ ]:
# -----------------------------
# Gradio App (app main)
# -----------------------------



with gr.Blocks(theme="soft", css=CSS) as demo:
    gr.HTML(css_inject)
    gr.Markdown("## Segment-Anything-like UI (Gradio Skeleton)")

    # Global state (per session)
    # {
    #   "workdir": str,
    #   "frame_paths": [str],
    #   "objects": { name: {"add":[(x,y,f)], "remove":[...]}, ... },
    #   "refs": {"Ref 1":[(x,y,f)], "Ref 2":[(x,y,f)]},
    #   "inference_state": predictor-specific state,
    #   "obj_name_to_id": { name: int },
    #   "next_obj_id": int,
    #   "obj_colors": { name: (r,g,b,a) }
    # }
    st = gr.State({
        "workdir": None,
        "frame_paths": [],
        "objects": {},
        "refs": {"Ref 1": [], "Ref 2": []},
        "inference_state": None,
        "obj_name_to_id": {},
        "next_obj_id": 1,
        "obj_colors": {},
        # propagation/mask storage
        "masks_per_frame": [],  # list of dicts: [{obj_id: mask_np}, ...]
        "overlay_paths": [],
        "propagated": False,
        "busy": False,
    })

    with gr.Row():
        # ---------------- Left: Toolbar ----------------
        with gr.Column(scale=1, elem_id="left_col"):
            gr.Markdown("### 🎛️ Toolbar")

            video = gr.Video(label="Upload a video")
            process_btn = gr.Button("Process (Extract Frames)")
            
            with gr.Accordion("Reference Objects (for coordinates only)", open=True):
                active_ref = gr.Dropdown(choices=["Ref 1", "Ref 2"], value="Ref 1", label="Active reference")
                clear_ref_btn = gr.Button("Clear points of active reference")

            with gr.Accordion("Objects to Segment", open=True):
                active_object = gr.Dropdown(choices=[], label="Active object", value=None)
                add_object_btn = gr.Button("Add object")
                remove_object_btn = gr.Button("Remove active object")

                point_mode = gr.Radio(
                    choices=["Add point", "Remove point"],
                    value="Add point",
                    label="Point mode"
                )
                clear_points_btn = gr.Button("Clear points of active object")

            track_btn = gr.Button("🚀 Track Objects")
            export_btn = gr.Button("Export Masks (.json)")
             # progress_log moved to the right column so toolbar stays compact

        # ---------------- Right: Viewer ----------------
        with gr.Column(scale=3, elem_id="right_col"):
            gr.Markdown("### 📹 Video & Frame Viewer")

            # Removed video_player to save vertical space and avoid duplicate playback UI
            with gr.Row():
                # 현재 프레임을 클릭 가능한 이미지로 표시
                frame_img = gr.Image(label="Click on the frame (coordinates captured here only)",
                                     interactive=True, type="pil", elem_id="frame_img")

            frame_slider = gr.Slider(0, 0, value=0, step=1, label="Frame index", interactive=True)

            # 클릭 좌표 표시용 텍스트
            click_info = gr.HTML("Click info will appear here.")

            # Progress log anchored at the bottom of this column
            progress_log = gr.Textbox(label="Progress log", lines=6)

    # -----------------------------
    # Utilities for mask overlay and colors
    # -----------------------------
    def _get_or_assign_color(state: Dict[str, Any], obj_name: str):
        if obj_name in state["obj_colors"]:
            return state["obj_colors"][obj_name]
        # pick a color from tab10 based on next_obj_id to have reproducible but distinct colors
        cmap = plt.get_cmap("tab10")
        idx = (state.get("next_obj_id", 1) - 1) % 10
        rgba = tuple(int(c * 255) for c in cmap(idx)[:3]) + (150,)  # semi-transparent alpha
        state["obj_colors"][obj_name] = rgba
        return rgba

    def _overlay_masks_on_image(state: Dict[str, Any], frame_path: str, masks_dict: Dict[int, np.ndarray], id_to_name: Dict[int, str], frame_idx: int = 0):
        """Overlay boolean masks onto the image using per-object colors and optionally draw object points.
        masks_dict: {obj_id: mask_np (H,W) boolean}
        id_to_name: {obj_id: obj_name}
        frame_idx: int - draw object points on top only when frame_idx == 0
        Returns path to saved overlay image (JPEG).
        """
        im = Image.open(frame_path).convert("RGBA")
        overlay = Image.new("RGBA", im.size, (0, 0, 0, 0))
        for obj_id, mask in masks_dict.items():
            if mask is None:
                continue
            obj_name = id_to_name.get(obj_id, f"Obj{obj_id}")
            color = _get_or_assign_color(state, obj_name)
            # build color image
            color_img = Image.new("RGBA", im.size, color)
            # Ensure mask is HxW uint8 0/255
            try:
                mask_img = Image.fromarray((mask.astype("uint8") * 255).astype("uint8"))
            except Exception:
                # fallback if mask already 0/255
                mask_img = Image.fromarray(mask.astype("uint8"))
            mask_img = mask_img.convert("L")
            # paste colored area onto overlay using mask
            overlay.paste(color_img, (0, 0), mask_img)
        # composite on top of original
        composed = Image.alpha_composite(im, overlay)
        # Convert to RGB for drawing points
        composed = composed.convert("RGB")

        # Draw object add/remove points on top of mask overlay only for frame 0
        if frame_idx == 0:
            draw = ImageDraw.Draw(composed)
            objs = state.get("objects", {})
            # For each mapped id -> name, draw that object's points that occur at frame 0
            for oid, name in id_to_name.items():
                buckets = objs.get(name, {"add": [], "remove": []})
                # positive points -> green circle, negative -> red X
                for (x, y, f) in buckets.get("add", []):
                    if f != 0:
                        continue
                    r = 6
                    draw.ellipse((int(x) - r, int(y) - r, int(x) + r, int(y) + r), outline=(0, 255, 0), width=2)
                for (x, y, f) in buckets.get("remove", []):
                    if f != 0:
                        continue
                    r = 6
                    draw.line((int(x) - r, int(y) - r, int(x) + r, int(y) + r), fill=(255, 0, 0), width=2)
                    draw.line((int(x) - r, int(y) + r, int(x) + r, int(y) - r), fill=(255, 0, 0), width=2)

        out_path = os.path.join(state["workdir"], f"overlay_masks_{uuid.uuid4().hex[:8]}.jpg")
        composed.save(out_path)
        return out_path

    def _normalize_mask_array(mask) -> np.ndarray:
        """Normalize various mask outputs into a 2D uint8 array with values 0 or 1.

        Accepts torch tensors, numpy arrays, or array-like masks that may have extra
        channel or singleton dimensions (e.g., (1, H, W), (1,1,H,W), (H,W,1), etc.).
        Returns a 2D numpy array dtype uint8 with values 0 or 1.
        Raises ValueError if mask cannot be converted to 2D.
        """
        # Convert to numpy
        try:
            import torch as _torch
            if _torch.is_tensor(mask):
                arr = mask.detach().cpu().numpy()
            else:
                arr = np.asarray(mask)
        except Exception:
            arr = np.asarray(mask)

        # Squeeze singleton dims until 2D or no singleton dims left
        arr = np.squeeze(arr)

        # If still has more than 2 dims, try to take the last two dims (best-effort)
        if arr.ndim > 2:
            try:
                arr = arr.reshape(arr.shape[-2], arr.shape[-1])
            except Exception:
                raise ValueError(f"Cannot normalize mask with shape {arr.shape}")

        if arr.ndim != 2:
            raise ValueError(f"Mask has wrong number of dims after normalization: {arr.ndim}")

        # Convert to boolean then uint8 (0/1)
        try:
            mask_bool = (arr > 0).astype('uint8')
        except Exception:
            mask_bool = (np.asarray(arr).astype('float32') > 0).astype('uint8')

        return mask_bool

    def _pil_from_path(path: str):
        """Safely load a PIL Image from a file path and convert to RGB. Returns None on failure."""
        if path is None:
            return None
        try:
            im = Image.open(path).convert("RGB")
            return im
        except Exception as e:
            print(f"[_pil_from_path] failed to open {path}: {e}")
            return None

    # -----------------------------
    # Callbacks
    # -----------------------------
    def on_process(video_data, state: Dict[str, Any]):
        """
        Extract frames and prime UI. Also initialize the predictor inference_state with the extracted frames.
        """
        if state.get("busy"):
            raise gr.Error("Busy processing. Wait until current operation finishes.")

        if video_data is None:
            raise gr.Error("Please upload a video first.")

        # Prepare unique workdir
        base_tmp = tempfile.gettempdir()
        workdir = os.path.join(base_tmp, f"gr_sam2_{uuid.uuid4().hex}")
        os.makedirs(workdir, exist_ok=True)

        # Save uploaded video to workdir
        if isinstance(video_data, dict) and "name" in video_data:
            video_path = video_data["name"]
        else:
            video_path = str(video_data)

        # Extract frames
        try:
            frame_paths = extract_frames(video_path, workdir)
        except subprocess.CalledProcessError as e:
            shutil.rmtree(workdir, ignore_errors=True)
            raise gr.Error("ffmpeg failed to extract frames. Ensure ffmpeg is installed and the video is valid.")

        if not frame_paths:
            shutil.rmtree(workdir, ignore_errors=True)
            raise gr.Error("No frames were extracted from the video.")

        # Reset state
        state["workdir"] = workdir
        state["frame_paths"] = frame_paths
        state["objects"] = {}
        state["refs"] = {"Ref 1": [], "Ref 2": []}
        state["obj_name_to_id"] = {}
        state["next_obj_id"] = 1
        state["obj_colors"] = {}
        state["masks_per_frame"] = [{} for _ in frame_paths]
        state["overlay_paths"] = [None for _ in frame_paths]
        state["propagated"] = False
        state["busy"] = False

        # Initialize predictor inference state using the extracted frames directory
        frames_dir = os.path.join(workdir, "frames")
        try:
            inference_state = predictor.init_state(video_path=frames_dir)
            state["inference_state"] = inference_state
        except Exception as e:
            # ignore predictor errors but report to user
            shutil.rmtree(workdir, ignore_errors=True)
            raise gr.Error(f"Failed to initialize predictor: {e}")

        # Initialize UI: frame slider and first frame
        first_frame = frame_paths[0]
        max_idx = len(frame_paths) - 1

        return (
            state,
            gr.update(value=_pil_from_path(first_frame)),
            gr.update(minimum=0, maximum=max_idx, value=0, interactive=True),
            gr.update(choices=[], value=None),
            "Ready. Frames extracted and predictor initialized."
        )

    process_btn.click(
        on_process,
        inputs=[video, st],
        outputs=[st, frame_img, frame_slider, active_object, progress_log]
    )

    def on_add_object(state: Dict[str, Any]):
        if state.get("busy"):
            return state, gr.update(choices=list(state.get("objects", {}).keys()), value=None)
        objs = state["objects"]
        # Create a new object name
        idx = 1
        while True:
            name = f"Object {idx}"
            if name not in objs:
                break
            idx += 1
        objs[name] = {"add": [], "remove": []}
        state["objects"] = objs
        # assign numeric id for predictor
        obj_id = state.get("next_obj_id", 1)
        state.setdefault("obj_name_to_id", {})[name] = obj_id
        state["next_obj_id"] = obj_id + 1
        # assign a color
        _ = _get_or_assign_color(state, name)
        return state, gr.update(choices=list(objs.keys()), value=name)

    add_object_btn.click(
        on_add_object,
        inputs=[st],
        outputs=[st, active_object]
    )

    def on_remove_active_object(state: Dict[str, Any], active_obj: str, frame_idx: int):
        if state.get("busy"):
            return state, gr.update(choices=list(state.get("objects", {}).keys()), value=active_obj), gr.update(value=None), "Busy: cannot remove object now.", gr.update(value="")
        objs = state["objects"]
        if active_obj in objs:
            # remove mapping for predictor if exists
            obj_map = state.get("obj_name_to_id", {})
            obj_id = obj_map.pop(active_obj, None)
            try:
                if obj_id is not None and state.get("inference_state") is not None:
                    # call predictor to remove object (best-effort)
                    predictor.remove_object(state["inference_state"], obj_id)
            except Exception:
                pass
            del objs[active_obj]
        state["objects"] = objs
        new_choices = list(objs.keys())
        new_value = new_choices[0] if new_choices else None

        # If no frames, just update dropdown and notify
        if not state.get("frame_paths"):
            return state, gr.update(choices=new_choices, value=new_value), gr.update(value=None), "Removed object.", gr.update(value="")

        # Redraw overlay for current frame to reflect removal
        fidx = int(frame_idx)
        fidx = max(0, min(fidx, len(state["frame_paths"]) - 1))
        frame_path = state["frame_paths"][fidx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
        out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
        im.save(out_path)

        # Return updated dropdown, refreshed image, a log message and clear click info
        return state, gr.update(choices=new_choices, value=new_value), gr.update(value=_pil_from_path(out_path)), f"Removed {active_obj}.", gr.update(value="")

    remove_object_btn.click(
        on_remove_active_object,
        inputs=[st, active_object, frame_slider],
        outputs=[st, active_object, frame_img, progress_log, click_info]
    )

    def on_clear_object_points(state: Dict[str, Any], active_obj: str, frame_idx: int):
        if state.get("busy"):
            return state, "Busy: cannot clear points now.", gr.update(value=None)
        # Clear points for the active object
        if active_obj and active_obj in state["objects"]:
            state["objects"][active_obj] = {"add": [], "remove": []}

        # If no frames available, just return and clear image
        if not state.get("frame_paths"):
            return state, "Cleared points for selected object.", gr.update(value=None)

        # Redraw overlay for current frame to reflect cleared points
        fidx = int(frame_idx)
        fidx = max(0, min(fidx, len(state["frame_paths"]) - 1))
        frame_path = state["frame_paths"][fidx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
        out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
        im.save(out_path)

        return state, "Cleared points for selected object.", gr.update(value=_pil_from_path(out_path))

    clear_points_btn.click(
        on_clear_object_points,
        inputs=[st, active_object, frame_slider],
        outputs=[st, progress_log, frame_img]
    )

    def on_clear_ref_points(state: Dict[str, Any], active_ref_name: str, frame_idx: int):
        if state.get("busy"):
            return state, f"Busy: cannot clear ref points now.", gr.update(value=None)
        # Clear points for the selected reference
        if active_ref_name in state["refs"]:
            state["refs"][active_ref_name] = []

        # If no frames available, clear image
        if not state.get("frame_paths"):
            return state, f"Cleared points for {active_ref_name}.", gr.update(value=None)

        # Redraw overlay for current frame to reflect cleared reference points
        fidx = int(frame_idx)
        fidx = max(0, min(fidx, len(state["frame_paths"]) - 1))
        frame_path = state["frame_paths"][fidx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
        out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
        im.save(out_path)

        return state, f"Cleared points for {active_ref_name}.", gr.update(value=_pil_from_path(out_path))

    clear_ref_btn.click(
        on_clear_ref_points,
        inputs=[st, active_ref, frame_slider],
        outputs=[st, progress_log, frame_img]
    )

    def on_frame_change(state: Dict[str, Any], frame_idx: int):
        # Update the clickable image to the chosen frame with overlays
        if not state["frame_paths"]:
            return gr.update(value=None)
        frame_idx = int(frame_idx)
        frame_idx = max(0, min(frame_idx, len(state["frame_paths"]) - 1))
        # remember last shown frame index in state so other callbacks can update the same frame
        state["last_frame_idx"] = frame_idx
        # If propagation ran and overlays exist, prefer overlay_paths
        overlay_paths = state.get("overlay_paths", [])
        if overlay_paths and overlay_paths[frame_idx]:
            return state, gr.update(value=_pil_from_path(overlay_paths[frame_idx]))

        frame_path = state["frame_paths"][frame_idx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], frame_idx)
        # Save to a temp path within workdir for display
        out_path = os.path.join(state["workdir"], f"overlay_{frame_idx:05d}.jpg")
        im.save(out_path)
        return state, gr.update(value=_pil_from_path(out_path))

    frame_slider.change(
        on_frame_change,
        inputs=[st, frame_slider],
        outputs=[st, frame_img]
    )

    def on_click_frame(evt: gr.SelectData,
                       state: Dict[str, Any],
                       active_obj: str,
                       point_mode_choice: str,
                       active_ref_name: str,
                       frame_idx: int):
        """
        Receive clicks ONLY from the frame image.
        Store them either in:
          - objects[active_obj]["add"/"remove"], or
          - refs[active_ref_name]
        Also call predictor.add_new_points_or_box for the active object and overlay predicted masks.
        """
        if state.get("busy"):
            return state, gr.update(), "Busy processing. Please wait until tracking completes."

        if not state["frame_paths"]:
            return state, gr.update(), "No video processed yet."

        x, y = evt.index  # (x, y) on displayed image
        fidx = int(frame_idx)

        # Decide target bucket
        updated_text = ""
        out_mask_overlay_path = None
        
        if active_obj:
            # debugpy.breakpoint()
            bucket = "add" if point_mode_choice == "Add point" else "remove"
            state["objects"].setdefault(active_obj, {"add": [], "remove": []})
            state["objects"][active_obj][bucket].append((x, y, fidx))
            updated_text = f"[{active_obj}] {bucket} point: (x={x}, y={y}, f={fidx})"

            # call predictor with this click (positive -> label 1, remove -> label 0)
            try:
                obj_map = state.get("obj_name_to_id", {})
                obj_id = obj_map.get(active_obj)
                if obj_id is None:
                    # assign id if missing
                    obj_id = state.get("next_obj_id", 1)
                    obj_map[active_obj] = obj_id
                    state["next_obj_id"] = obj_id + 1
                    state["obj_name_to_id"] = obj_map
                    _get_or_assign_color(state, active_obj)

                lbl = 1 if bucket == "add" else 0
                pts = np.array([[x, y]], dtype=np.float32)
                labels = np.array([lbl], dtype=np.int32)
                # predictor expects the inference_state stored in state
                if state.get("inference_state") is not None:
                    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
                        inference_state=state["inference_state"],
                        frame_idx=fidx,
                        obj_id=obj_id,
                        points=pts,
                        labels=labels,
                    )

                    # out_mask_logits is a torch tensor at video resolution for returned obj_ids
                    masks_dict = {}
                    if out_mask_logits is not None:
                        # convert to numpy boolean masks per returned object id
                        try:
                            masks_np = out_mask_logits.cpu().numpy()
                        except Exception:
                            masks_np = out_mask_logits.numpy()

                        # out_obj_ids aligns with masks_np in axis 0
                        for i, oid in enumerate(out_obj_ids):
                            try:
                                mask_raw = masks_np[i]
                                mask_norm = _normalize_mask_array(mask_raw)
                                masks_dict[oid] = mask_norm
                            except Exception as e:
                                print(f"[on_click_frame] unexpected mask shape for oid={oid}: {e}")
                                continue

                    # Build id->name mapping from state
                    id_to_name = {v: k for k, v in state.get("obj_name_to_id", {}).items()}
                    # overlay masks on top of original frame
                    frame_path = state["frame_paths"][fidx]
                    if masks_dict:
                        # pass frame index so overlay function can decide whether to draw points
                        out_mask_overlay_path = _overlay_masks_on_image(state, frame_path, masks_dict, id_to_name, frame_idx=fidx)
                        # store masks into masks_per_frame for later use
                        mpf = state.get("masks_per_frame", [{} for _ in state.get("frame_paths", [])])
                        for oid, m in masks_dict.items():
                            mpf[fidx][oid] = m
                        state["masks_per_frame"] = mpf

            except Exception as e:
                updated_text += f" (predictor error: {pts}, {labels}, {e})"
        else:
            # If no object is active, record to reference
            state["refs"].setdefault(active_ref_name, [])
            state["refs"][active_ref_name].append((x, y, fidx))
            updated_text = f"[{active_ref_name}] point: (x={x}, y={y}, f={fidx})"

        # Redraw overlay for current frame if predictor didn't produce a mask overlay
        if out_mask_overlay_path is None:
            frame_path = state["frame_paths"][fidx]
            im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
            out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
            im.save(out_path)
            image_update = gr.update(value=_pil_from_path(out_path))
        else:
            image_update = gr.update(value=_pil_from_path(out_mask_overlay_path))

        info_html = f"<code>{updated_text}</code>"
        return state, image_update, info_html

    frame_img.select(
        on_click_frame,
        inputs=[st, active_object, point_mode, active_ref, frame_slider],
        outputs=[st, frame_img, click_info]
    )

    def track_objects(state: Dict[str, Any]):
        """
        Run predictor.propagate_in_video to get per-frame segmentation masks for all objects.
        This operation can be slow; set state["busy"] during processing to prevent other interactions.
        While propagation yields per-frame masks, build overlay for that frame and immediately
        update the displayed frame image and slider so the user sees progressive results.
        After propagation, ensure overlays exist for all frames.
        """
        if not state.get("frame_paths"):
            yield "No frames to process.", gr.update(value=None), gr.update(value=0)
            return
        if state.get("busy"):
            yield "Already running.", gr.update(value=None), gr.update(value=0)
            return

        n = len(state["frame_paths"])
        # determine which frame is currently shown so we can update it during/after propagation
        last_idx = int(state.get("last_frame_idx", 0)) if state.get("frame_paths") else 0

        state["busy"] = True
        # initialize overlay_paths and ensure masks_per_frame has correct length
        state.setdefault("overlay_paths", [None for _ in range(n)])
        state.setdefault("masks_per_frame", [{} for _ in range(n)])

        yield "Propagation started...", gr.update(value=_pil_from_path(state["frame_paths"][last_idx])), gr.update(value=last_idx)

        # Collect per-frame results from the predictor generator and update display per-frame
        video_segments = {}
        id_to_name = {v: k for k, v in state.get("obj_name_to_id", {}).items()}
        try:
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(state["inference_state"]):
                # build dict of obj_id -> numpy boolean mask
                frame_dict = {}
                for i, out_obj_id in enumerate(out_obj_ids):
                    try:
                        # out_mask_logits[i] may be a torch tensor or numpy array
                        raw = out_mask_logits[i]
                        try:
                            import torch as _torch
                            if _torch.is_tensor(raw):
                                raw_np = raw.cpu().numpy()
                            else:
                                raw_np = np.asarray(raw)
                        except Exception:
                            raw_np = np.asarray(raw)
                        norm = _normalize_mask_array(raw_np)
                        frame_dict[int(out_obj_id)] = norm
                    except Exception as e:
                        print(f"[track_objects] failed to convert mask for frame {out_frame_idx}, obj {out_obj_id}: {e}")

                out_frame_idx = int(out_frame_idx)
                video_segments[out_frame_idx] = frame_dict

                # store into state masks_per_frame
                if 0 <= out_frame_idx < n:
                    state["masks_per_frame"][out_frame_idx] = frame_dict

                # Build overlay for this frame immediately
                frame_path = state["frame_paths"][out_frame_idx]
                if frame_dict:
                    try:
                        overlay_path = _overlay_masks_on_image(state, frame_path, frame_dict, id_to_name, frame_idx=out_frame_idx)
                    except Exception as e:
                        print(f"[track_objects] overlay creation failed for frame {out_frame_idx}: {e}")
                        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], out_frame_idx)
                        overlay_path = os.path.join(state["workdir"], f"overlay_{out_frame_idx:05d}.jpg")
                        im.save(overlay_path)
                else:
                    im = draw_points_on_frame(frame_path, state["objects"], state["refs"], out_frame_idx)
                    overlay_path = os.path.join(state["workdir"], f"overlay_{out_frame_idx:05d}.jpg")
                    im.save(overlay_path)

                # store overlay path
                state["overlay_paths"][out_frame_idx] = overlay_path

                # Immediately update displayed image and slider to this frame
                try:
                    pil_img = _pil_from_path(overlay_path)
                except Exception:
                    pil_img = _pil_from_path(frame_path)

                yield f"Propagated frame {out_frame_idx+1}/{n}", gr.update(value=pil_img), gr.update(value=out_frame_idx)

        except Exception as e:
            state["busy"] = False
            yield f"Propagation failed: {e}", gr.update(value=_pil_from_path(state["frame_paths"][last_idx])), gr.update(value=last_idx)
            return

        # build overlays for any frames that did not receive masks during propagation
        for f in range(n):
            if state["overlay_paths"][f] is None:
                masks_dict = state["masks_per_frame"][f]
                frame_path = state["frame_paths"][f]
                if masks_dict:
                    try:
                        state["overlay_paths"][f] = _overlay_masks_on_image(state, frame_path, masks_dict, id_to_name, frame_idx=f)
                    except Exception as e:
                        print(f"[track_objects] overlay creation failed for frame {f}: {e}")
                        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], f)
                        out_path = os.path.join(state["workdir"], f"overlay_{f:05d}.jpg")
                        im.save(out_path)
                        state["overlay_paths"][f] = out_path
                else:
                    im = draw_points_on_frame(frame_path, state["objects"], state["refs"], f)
                    out_path = os.path.join(state["workdir"], f"overlay_{f:05d}.jpg")
                    im.save(out_path)
                    state["overlay_paths"][f] = out_path

        state["propagated"] = True
        state["busy"] = False

        final_display = state["overlay_paths"][last_idx] if state["overlay_paths"] and 0 <= last_idx < len(state["overlay_paths"]) else state["frame_paths"][last_idx]
        yield f"Done. Overlays built for all frames. {state['workdir']}", gr.update(value=_pil_from_path(final_display)), gr.update(value=last_idx)

    track_btn.click(
        track_objects,
        inputs=[st],
        outputs=[progress_log, frame_img, frame_slider]
    )
    
    def on_export_masks(state: Dict[str, Any]):
        """TODO"""
        return state, f"Exported masks End"

    export_btn.click(
        on_export_masks,
        inputs=[st],
        outputs=[st, progress_log]
    )

    # debugpy.listen(5678)
    # print("Attach debugger to port 5678 and continue")
    # debugpy.wait_for_client()  # 디버거가 연결될 때까지 대기
    # print("Attached...")
    demo.launch()